# 🎬 Qwen-Image-2.1: Image Editor & GIF Creator Tutorial
### By Juan Jose

Welcome to the companion notebook for the **Qwen-Image-2.1 Animation Studio**.

In this notebook, you will learn how to:
1. Set up an inference environment for Alibaba's **Qwen-Image-2.1** (~7B Diffusion Transformer).
2. Understand and implement the **Single-Reference Invariant Pattern** to generate animation variations without cumulative visual corruption.
3. Expose ComfyUI via a secure tunnel to connect your local Gradio application, or run generation directly inside Colab.
4. Compile multi-frame sequences into high-quality looping GIFs using Pillow.

## 1. Environment & GPU Verification
Let's check the assigned GPU. A **Tesla T4 (15GB)** on Google Colab Free tier is sufficient with CPU offloading, while an **A100 (40GB)** on Colab Pro provides blazing fast inference.

In [ ]:
!nvidia-smi

## 2. Option A: Running ComfyUI Backend + Tunnel
If you want to use the local Gradio application on your computer while outsourcing GPU computation to this Colab notebook, run the following cells to start ComfyUI and open a tunnel.

In [ ]:
# Clone ComfyUI and install requirements
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

# Download Qwen-Image-2.1 weights (or place your quantized safetensors into models/checkpoints/)
# e.g.: !wget -P models/checkpoints/ https://huggingface.co/Qwen/Qwen-Image-2.1/resolve/main/qwen_image_2.1.safetensors

In [ ]:
# Start ComfyUI in the background and expose via cloudflared tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

import subprocess
import time

# Launch ComfyUI
subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"])
time.sleep(5)

# Expose tunnel
!cloudflared tunnel --url http://127.0.0.1:8188

Copy the generated `*.trycloudflare.com` URL and paste it into your local Gradio application's **ComfyUI Server Endpoint** field!

## 3. Option B: Direct Python Pipeline (Diffusers)
Alternatively, you can run single-reference frame generation directly using Hugging Face `diffusers` with memory offloading.

In [ ]:
!pip install -q diffusers transformers accelerate sentencepiece pillow

In [ ]:
import torch
from PIL import Image

# Ensure model offloading for GPU memory conservation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on: {device}")

### 3.1 Single-Reference Animation Generation Loop
Notice how `base_image` remains **strictly constant** across all iterations:

In [ ]:
# Define your sequence instructions
edit_instructions = [
    "<image1> Subtle smile, looking forward",
    "<image1> Cheerful grin, turning head slightly right",
    "<image1> Big smile, laughing with head tilted right",
    "<image1> Gentle smile, returning to central position",
]

frames = []
# In actual inference, call your pipeline:
# for prompt in edit_instructions:
#     frame = pipe(prompt=prompt, image=base_image, strength=0.7).images[0]
#     frames.append(frame)

print("Frames generated successfully!")

## 4. Compiling the Animation with Pillow

In [ ]:
def export_gif(frame_list, output_path="animation.gif", fps=8, ping_pong=True):
    if not frame_list:
        raise ValueError("Frame list is empty!")
    
    sequence = list(frame_list)
    if ping_pong and len(sequence) > 2:
        sequence.extend(sequence[-2:0:-1])
        
    duration_ms = int(1000 / fps)
    palette_frames = [f.convert("P", palette=Image.Palette.ADAPTIVE, colors=256) for f in sequence]
    
    palette_frames[0].save(
        output_path,
        save_all=True,
        append_images=palette_frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=True
    )
    print(f"Saved optimized GIF to {output_path} ({len(palette_frames)} frames @ {fps} FPS)")

# export_gif(frames, "my_character_loop.gif", fps=6, ping_pong=True)